# Data Consumption — Metadata Semantic Search

Natural-language queries over sound-cymatics metadata using
**all-MiniLM-L6-v2** text embeddings (384-dim) in the Milvus
`sound_text_embeddings` collection.

Prerequisites:
- Milvus running (`docker compose up -d milvus`)
- Text embeddings ingested (orchestrator → option 6)

## Environment setup

In [ ]:
from pathlib import Path
import os
import sys

_here = Path.cwd().resolve()
_root = next(
    (p for p in [_here, *_here.parents]
     if (p / "docker-compose.yml").is_file() and (p / "orchestrate.py").is_file()),
    None,
)
if _root is None:
    raise RuntimeError("Repo root not found — open the notebook from the BDM-Cymatics tree")

PROJECT_ROOT = str(_root)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(_root / ".env", override=False)

print("PROJECT_ROOT =", PROJECT_ROOT)

## Constants

In [ ]:
import numpy as np

TOP_K = 5

## Connect to Milvus

In [ ]:
from pymilvus import MilvusClient

MILVUS_URI = os.environ.get("MILVUS_URI", "http://localhost:19530")
milvus_client = MilvusClient(uri=MILVUS_URI)
print(f"Milvus connected: {MILVUS_URI}")
print(f"Collections: {milvus_client.list_collections()}")

## Compute text embedding — all-MiniLM-L6-v2 (384-dim)

In [ ]:
from sentence_transformers import SentenceTransformer

text_model = SentenceTransformer("all-MiniLM-L6-v2")
print("all-MiniLM-L6-v2 loaded")

def compute_text_embedding(text):
    # Compute a 384-dim text embedding from a query string.
    emb = text_model.encode(text, normalize_embeddings=True)
    return emb.astype(np.float32)

## Search Milvus — embed query and find nearest neighbours

In [ ]:
TEXT_COLLECTION = "sound_text_embeddings"

def search_metadata(query, top_k=TOP_K):
    # Embed the query and search Milvus for matching recordings.
    print(f"  Computing text embedding for: \"{query}\"")
    embedding = compute_text_embedding(query)

    results = milvus_client.search(
        collection_name=TEXT_COLLECTION,
        data=[embedding.tolist()],
        limit=top_k,
        output_fields=["uuid", "category", "source", "peak_frequency_hz", "description_text"],
        search_params={"metric_type": "COSINE", "params": {"nprobe": 16}},
    )
    hits = results[0] if results else []
    print(f"  Found {len(hits)} matching recording(s).")
    return hits

## Display results — format Milvus text-embedding similarity hits

In [ ]:
def display_results(results, query):
    # Print search results.
    print(f"\n{'=' * 62}")
    print(f"  Metadata Search — Top {len(results)}")
    print(f"  Query: \"{query}\"")
    print(f"{'─' * 62}")
    if not results:
        print("  No matching recordings found.")
        print(f"{'=' * 62}\n")
        return
    for i, hit in enumerate(results):
        e = hit["entity"]
        print(f"  {i+1}. Similarity: {hit['distance']:.4f}")
        print(f"     UUID:       {e.get('uuid', '?')}")
        print(f"     Category:   {e.get('category', '') or '—'}")
        print(f"     Source:      {e.get('source', '') or '—'}")
        print(f"     Peak freq:  {e.get('peak_frequency_hz', 0):.0f} Hz")
        desc = e.get("description_text", "") or "—"
        print(f"     Description: {desc[:80]}")
        print()
    print(f"{'=' * 62}\n")

## Run search — example queries

In [ ]:
queries = [
    "what frequency do rain sounds have?",
    "which sounds are highly harmonic?",
    "low frequency tonal sounds",
    "complex noise-like high frequency recordings",
]

for q in queries:
    results = search_metadata(q)
    display_results(results, q)